# 01 — Full Fine-Tuning vs LoRA on Banking77

Fine-tunes `bert-base-uncased` for 77-class intent classification two ways — updating every weight,
and updating only LoRA adapters (`r=8`, via HuggingFace `peft`) — and measures what the second
approach costs and saves.

**Protocol.** Both arms train to convergence rather than for a matched number of epochs: each runs
with early stopping on a held-out validation split and reports its best checkpoint on the test set.
Matching epochs across the two would understate LoRA, which optimises a much smaller parameter space
and needs more steps to reach the same place. Steps-to-convergence is recorded as a result in its own
right.

**Runtime:** GPU required (Colab or Kaggle). Roughly 30–45 minutes end to end.

## Setup

In [ ]:
!pip install -q transformers "datasets<4.0" peft accelerate evaluate torch scikit-learn
# Kaggle's base image ships torchao 0.10, which peft version-checks and rejects when injecting
# LoRA adapters. This project never uses torchao, so removing it sidesteps the check.
!pip uninstall -y -q torchao

In [ ]:
import gc
import json
import os
import time

import evaluate
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

BATCH_SIZE = 16
COLS = ["input_ids", "attention_mask", "labels"]

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print("WARNING: no GPU detected — switch to a GPU runtime before running the training cells.")

# Repo layout puts results/ next to notebooks/; hosted runtimes (Kaggle, Colab) have no
# parent dir to write to, so fall back to a local results/ and download it from the output panel.
RESULTS_DIR = "../results" if os.path.isdir("../results") else "results"
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Writing results to:", os.path.abspath(RESULTS_DIR))

## 1. Data

[Banking77](https://huggingface.co/datasets/PolyAI/banking77): ~10k train / ~3k test online-banking
customer queries, labelled with 77 fine-grained intents. Many classes are near-neighbours
("card_not_working" vs "card_payment_not_recognised"), which is what makes 77-way separation
non-trivial for a base-size encoder.

In [ ]:
dataset = load_dataset("PolyAI/banking77", trust_remote_code=True)
label_names = dataset["train"].features["label"].names
num_labels = len(label_names)
print(dataset)
print(f"Classes: {num_labels}")
print("First 10 labels:", label_names[:10])

In [ ]:
MODEL_NAME = "bert-base-uncased"
SEED = 42
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=64)


# padding="max_length" makes every batch fixed-size, so Trainer needs no data collator --
# which also sidesteps the tokenizer= -> processing_class= rename across transformers versions.
tokenized = dataset.map(tokenize_fn, batched=True)
tokenized = tokenized.rename_column("label", "labels")

# Banking77 ships train/test only. Early stopping and best-checkpoint selection have to run
# against data the final number is NOT reported on, so carve a stratified validation split
# out of train and leave test untouched until the final evaluate().
split = tokenized["train"].train_test_split(
    test_size=0.1, seed=SEED, stratify_by_column="labels"
)
train_ds = split["train"].with_format("torch", columns=COLS)
val_ds = split["test"].with_format("torch", columns=COLS)
test_ds = tokenized["test"].with_format("torch", columns=COLS)

print(f"Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}")
n_gpu = max(torch.cuda.device_count(), 1)
print(f"GPUs visible: {torch.cuda.device_count()} -> effective train batch = {BATCH_SIZE * n_gpu}")

## 2. Shared helpers

In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "macro_f1": f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"],
    }


def count_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total


def reset_peak_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        for i in range(torch.cuda.device_count()):
            torch.cuda.reset_peak_memory_stats(i)


def peak_memory_mb():
    """Highest per-GPU allocation. Kaggle's T4 x2 runs DataParallel, so device 0 alone
    under-reports; the max across devices is what a single card would need to hold."""
    if not torch.cuda.is_available():
        return None
    return max(
        torch.cuda.max_memory_allocated(i) for i in range(torch.cuda.device_count())
    ) / (1024 ** 2)


def epoch_history(trainer):
    """Per-epoch eval metrics pulled out of Trainer's log history."""
    return [
        {
            "epoch": round(rec["epoch"], 2),
            "eval_loss": rec.get("eval_loss"),
            "accuracy": rec.get("eval_accuracy"),
            "macro_f1": rec.get("eval_macro_f1"),
        }
        for rec in trainer.state.log_history
        if "eval_accuracy" in rec
    ]


def release_cuda():
    """Call immediately after `del`-ing the model/trainer names in the calling scope.
    The `del` has to happen at the call site: passing objects into a helper and deleting
    them there only drops the helper's own reference, so nothing is actually freed."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 3. Full fine-tuning (baseline)

All ~110M parameters trainable. Learning rate 2e-5 — the standard range for full BERT fine-tuning;
higher values destabilise the pretrained weights.

In [ ]:
full_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels
).to(device)

full_trainable, full_total = count_trainable_params(full_model)
print(f"[full] Trainable: {full_trainable:,} / {full_total:,} "
      f"({100 * full_trainable / full_total:.2f}%)")

full_args = TrainingArguments(
    output_dir="./full_finetune_out",
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    num_train_epochs=10,
    warmup_ratio=0.06,          # a fresh 77-way head at this LR diverges without it
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

full_trainer = Trainer(
    model=full_model,
    args=full_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

reset_peak_memory()
start = time.time()
full_trainer.train()
full_time = time.time() - start
full_mem = peak_memory_mb()
full_steps = full_trainer.state.global_step
full_epochs = full_trainer.state.epoch

print(f"[full] Stopped after {full_epochs:.0f} epochs / {full_steps} steps in {full_time:.1f}s")
if full_mem:
    print(f"[full] Peak GPU memory: {full_mem:.1f} MB")

In [ ]:
full_history = epoch_history(full_trainer)
full_test = full_trainer.evaluate(test_ds)
print("[full] Test:", {k: round(v, 4) for k, v in full_test.items() if isinstance(v, float)})

# Release the baseline before building the LoRA model. Without this the LoRA section's peak-memory
# reading includes the still-resident 110M-parameter model and its Adam state, which inverts the
# comparison the notebook exists to make.
del full_model, full_trainer
release_cuda()
print("Freed baseline model.",
      f"Allocated now: {torch.cuda.memory_allocated() / 1024 ** 2:.1f} MB"
      if torch.cuda.is_available() else "")

## 4. LoRA fine-tuning (`peft`, r=8)

Base weights frozen; each targeted projection gets a rank-8 update `ΔW = (alpha/r)·BA`.
`target_modules=["query", "value"]` follows the original LoRA paper's attention-only setting.
Learning rate 2e-4 — an order of magnitude above the full fine-tune, since the adapters start at
zero and have far fewer parameters to move.

In [ ]:
lora_base = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,   # also unfreezes the classification head via modules_to_save
    r=8,
    lora_alpha=16,                # scaling applied is alpha/r = 2
    lora_dropout=0.1,
    target_modules=["query", "value"],
)

lora_model = get_peft_model(lora_base, lora_config).to(device)
lora_model.print_trainable_parameters()
lora_trainable, lora_total = count_trainable_params(lora_model)

In [ ]:
lora_args = TrainingArguments(
    output_dir="./lora_out",
    learning_rate=2e-4,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    num_train_epochs=20,
    warmup_ratio=0.06,          # a fresh 77-way head at this LR diverges without it
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

lora_trainer = Trainer(
    model=lora_model,
    args=lora_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
)

reset_peak_memory()
start = time.time()
lora_trainer.train()
lora_time = time.time() - start
lora_mem = peak_memory_mb()
lora_steps = lora_trainer.state.global_step
lora_epochs = lora_trainer.state.epoch

print(f"[lora] Stopped after {lora_epochs:.0f} epochs / {lora_steps} steps in {lora_time:.1f}s")
if lora_mem:
    print(f"[lora] Peak GPU memory: {lora_mem:.1f} MB")

In [ ]:
lora_history = epoch_history(lora_trainer)
lora_test = lora_trainer.evaluate(test_ds)
print("[lora] Test:", {k: round(v, 4) for k, v in lora_test.items() if isinstance(v, float)})

## 5. Comparison

In [ ]:
comparison = pd.DataFrame([
    {
        "method": "full_finetune",
        "trainable_params": full_trainable,
        "trainable_pct": round(100 * full_trainable / full_total, 3),
        "accuracy": full_test["eval_accuracy"],
        "macro_f1": full_test["eval_macro_f1"],
        "train_time_s": round(full_time, 1),
        "peak_mem_mb": round(full_mem, 1) if full_mem else None,
        "epochs_run": round(full_epochs, 1),
        "steps_run": full_steps,
    },
    {
        "method": "lora_r8",
        "trainable_params": lora_trainable,
        "trainable_pct": round(100 * lora_trainable / lora_total, 3),
        "accuracy": lora_test["eval_accuracy"],
        "macro_f1": lora_test["eval_macro_f1"],
        "train_time_s": round(lora_time, 1),
        "peak_mem_mb": round(lora_mem, 1) if lora_mem else None,
        "epochs_run": round(lora_epochs, 1),
        "steps_run": lora_steps,
    },
])
comparison

### The trade-off

Neither the accuracy nor the parameter count means much alone. What LoRA is worth is the ratio
between them: the share of baseline accuracy retained against the share of parameters, time and
memory spent to retain it. The cell below computes that and emits the README table.

In [ ]:
f = comparison.iloc[0]
l = comparison.iloc[1]

tradeoff = {
    "accuracy_full": round(f["accuracy"], 4),
    "accuracy_lora": round(l["accuracy"], 4),
    "accuracy_gap_pts": round(100 * (f["accuracy"] - l["accuracy"]), 2),
    "accuracy_retained_pct": round(100 * l["accuracy"] / f["accuracy"], 2),
    "param_fraction_pct": round(100 * l["trainable_params"] / f["trainable_params"], 4),
    "param_reduction_x": round(f["trainable_params"] / l["trainable_params"], 1),
    "time_ratio": round(l["train_time_s"] / f["train_time_s"], 2),
    "memory_ratio": round(l["peak_mem_mb"] / f["peak_mem_mb"], 2) if f["peak_mem_mb"] else None,
    "steps_ratio": round(l["steps_run"] / f["steps_run"], 2),
}

print(
    f"LoRA retained {tradeoff['accuracy_retained_pct']}% of full fine-tuning accuracy "
    f"({tradeoff['accuracy_lora']:.4f} vs {tradeoff['accuracy_full']:.4f}, a gap of "
    f"{tradeoff['accuracy_gap_pts']} points) while training "
    f"{tradeoff['param_fraction_pct']}% of the parameters "
    f"({tradeoff['param_reduction_x']}x fewer), using {tradeoff['memory_ratio']}x peak GPU memory "
    f"and {tradeoff['time_ratio']}x wall-clock time across {tradeoff['steps_ratio']}x the "
    f"optimizer steps."
)

print("\n--- paste into README ---\n")
print("| Method | Trainable params | % of total | Accuracy | Macro F1 | Epochs | Steps | Train time (s) | Peak GPU mem (MB) |")
print("|---|---|---|---|---|---|---|---|---|")
for _, r in comparison.iterrows():
    print(
        f"| {r['method']} | {r['trainable_params']:,} | {r['trainable_pct']}% | "
        f"{r['accuracy']:.4f} | {r['macro_f1']:.4f} | {r['epochs_run']:.0f} | {r['steps_run']} | "
        f"{r['train_time_s']} | {r['peak_mem_mb']} |"
    )

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(19, 4))
colors = ["#2a78d6", "#eb6834"]

axes[0].bar(comparison["method"], comparison["trainable_params"], color=colors)
axes[0].set_yscale("log")
axes[0].set_title("Trainable parameters (log)")

axes[1].bar(comparison["method"], comparison["accuracy"], color=colors)
axes[1].set_ylim(0, 1)
axes[1].set_title("Test accuracy")

axes[2].bar(comparison["method"], comparison["train_time_s"], color=colors)
axes[2].set_title("Training time (s)")

axes[3].bar(comparison["method"], comparison["peak_mem_mb"], color=colors)
axes[3].set_title("Peak GPU memory (MB)")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/01_baseline_comparison.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot([h["epoch"] for h in full_history], [h["accuracy"] for h in full_history],
        marker="o", color="#2a78d6", label="Full fine-tune")
ax.plot([h["epoch"] for h in lora_history], [h["accuracy"] for h in lora_history],
        marker="o", color="#eb6834", label="LoRA r=8")
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation accuracy")
ax.set_title("Convergence: LoRA needs more epochs to reach the same place")
ax.legend()
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/01_training_curves.png", dpi=150)
plt.show()

In [ ]:
comparison.to_csv(f"{RESULTS_DIR}/01_baseline_comparison.csv", index=False)
with open(f"{RESULTS_DIR}/01_baseline_comparison.json", "w") as fh:
    json.dump(comparison.to_dict(orient="records"), fh, indent=2)
with open(f"{RESULTS_DIR}/01_tradeoff.json", "w") as fh:
    json.dump(tradeoff, fh, indent=2)
with open(f"{RESULTS_DIR}/01_training_curves.json", "w") as fh:
    json.dump({"full_finetune": full_history, "lora_r8": lora_history}, fh, indent=2)

print("Wrote:")
for name in ["01_baseline_comparison.csv", "01_baseline_comparison.json",
             "01_tradeoff.json", "01_training_curves.json",
             "01_baseline_comparison.png", "01_training_curves.png"]:
    print(" ", os.path.join(RESULTS_DIR, name))